# Library Replicate Comparison

This notebook compares LibA and LibB functional effects, separated by func-1 and func-2, across all entry conditions (2-3, 2-6, and mix).

## Import modules

In [1]:
import pandas as pd
import altair as alt

# Enable data transformer to handle large datasets
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

## Configuration

In [2]:
# Define entry conditions and their corresponding selections
entry_conditions = {
    "2-3 entry": {
        "LibA-func-1": "LibA-250521-293-2-3-func-1",
        "LibA-func-2": "LibA-250521-293-2-3-func-2",
        "LibB-func-1": "LibB-250521-293-2-3-func-1",
        "LibB-func-2": "LibB-250521-293-2-3-func-2",
    },
    "2-6 entry": {
        "LibA-func-1": "LibA-250521-293-2-6-func-1",
        "LibA-func-2": "LibA-250521-293-2-6-func-2",
        "LibB-func-1": "LibB-250521-293-2-6-func-1",
        "LibB-func-2": "LibB-250521-293-2-6-func-2",
    },
    "mix entry": {
        "LibA-func-1": "LibA-250521-293-mix-func-1",
        "LibA-func-2": "LibA-250521-293-mix-func-2",
        "LibB-func-1": "LibB-250521-293-mix-func-1",
        "LibB-func-2": "LibB-250521-293-mix-func-2",
    },
}

# Times seen threshold
init_times_seen = 2

## Load functional effects data

Load the functional effects for each selection from the individual CSV files.

In [3]:
def load_func_effects(entry_conditions):
    """Load functional effects for all selections."""
    dfs = []
    for entry_condition, selections in entry_conditions.items():
        for label, selection in selections.items():
            df = pd.read_csv(f"../results/func_effects/by_selection/{selection}_func_effects.csv")
            df["selection"] = selection
            df["entry_condition"] = entry_condition
            df["label"] = label
            dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

# Load data
func_effects = load_func_effects(entry_conditions)

# Convert times_seen to Int64
func_effects["times_seen"] = func_effects["times_seen"].astype("Int64")

print(f"Loaded {len(func_effects):,} rows of functional effects data")
print(f"Entry conditions: {func_effects['entry_condition'].unique()}")
print(f"Labels: {sorted(func_effects['label'].unique())}")

Loaded 117,348 rows of functional effects data
Entry conditions: ['2-3 entry' '2-6 entry' 'mix entry']
Labels: ['LibA-func-1', 'LibA-func-2', 'LibB-func-1', 'LibB-func-2']


## Library comparison plots for all variants

Create scatter plots comparing LibA vs LibB for func-1 and func-2 separately, across all entry conditions.

In [4]:
print(f"Library comparison scatter plots for times_seen filter of {init_times_seen}\n")

# Create mutation identifier
func_effects["mutation"] = (
    func_effects["wildtype"] + 
    func_effects["site"].astype(str) + 
    func_effects["mutant"]
)

# Filter by times_seen
func_effects_filtered = func_effects.query("times_seen >= @init_times_seen")

# Mutation selection for interactive highlighting
mutation_selection = alt.selection_point(
    fields=["mutation"],
    on="mouseover",
    empty=False,
)

# Create comparison plots for each entry condition and func type
plots = []

for entry_condition in ["2-3 entry", "2-6 entry", "mix entry"]:
    entry_plots = []
    
    for func_type in ["func-1", "func-2"]:
        # Get data for LibA and LibB for this func type
        libA_label = f"LibA-{func_type}"
        libB_label = f"LibB-{func_type}"
        
        df = (
            func_effects_filtered
            .query("entry_condition == @entry_condition")
            .query("label in [@libA_label, @libB_label]")
            .pivot_table(
                index="mutation",
                columns="label",
                values="functional_effect"
            )
            .reset_index()
        )
        
        # Check if we have both columns
        if libA_label not in df.columns or libB_label not in df.columns:
            print(f"Skipping {entry_condition} {func_type}: missing data")
            continue
        
        # Drop NaN values and calculate correlation
        df_clean = df[[libA_label, libB_label]].dropna()
        n = len(df_clean)
        r = df_clean.corr().values[1, 0] if n > 0 else 0
        
        print(f"{entry_condition} - {func_type}: n={n}, r={r:.3f}")
        
        if n == 0:
            continue
        
        # Create scatter plot
        chart = (
            alt.Chart(df)
            .add_params(mutation_selection)
            .encode(
                alt.X(
                    libA_label,
                    scale=alt.Scale(nice=False, padding=4),
                    title=f"LibA {func_type}"
                ),
                alt.Y(
                    libB_label,
                    scale=alt.Scale(nice=False, padding=4),
                    title=f"LibB {func_type}"
                ),
                size=alt.condition(mutation_selection, alt.value(80), alt.value(30)),
                color=alt.condition(
                    mutation_selection, alt.value("red"), alt.value("black")
                ),
                opacity=alt.condition(
                    mutation_selection, alt.value(1), alt.value(0.25)
                ),
                tooltip=[
                    "mutation",
                    alt.Tooltip(libA_label, format=".3f"),
                    alt.Tooltip(libB_label, format=".3f"),
                ],
            )
            .mark_circle()
            .properties(
                width=250,
                height=250,
                title=alt.TitleParams(
                    f"{entry_condition}\nR = {r:.2f}, N = {n}",
                    fontSize=12,
                    fontWeight="normal",
                ),
            )
        )
        
        entry_plots.append(chart)
    
    if entry_plots:
        plots.append(alt.hconcat(*entry_plots))

# Display all plots
if plots:
    final_chart = (
        alt.vconcat(*plots)
        .configure_axis(grid=False)
        .properties(title="Library Replicate Comparison: LibA vs LibB (All Variants)")
    )
    display(final_chart)
else:
    print("No plots generated")

Library comparison scatter plots for times_seen filter of 2

2-3 entry - func-1: n=6835, r=0.828


2-3 entry - func-2: n=6802, r=0.826


2-6 entry - func-1: n=6835, r=0.897
2-6 entry - func-2: n=6802, r=0.894


mix entry - func-1: n=6835, r=0.885


mix entry - func-2: n=6802, r=0.885


alt.VConcatChart(...)

## Load single-mutant data

In [5]:
def load_func_effects_singlemut(entry_conditions):
    """Load functional effects from single-mutant variants only."""
    dfs = []
    for entry_condition, selections in entry_conditions.items():
        for label, selection in selections.items():
            df = pd.read_csv(f"../results/func_effects/by_selection/{selection}_func_effects.csv")
            # Use single-mutant columns
            df = df[["site", "wildtype", "mutant", "functional_effect_singlemut", "times_seen_singlemut"]].copy()
            df.rename(
                columns={
                    "functional_effect_singlemut": "functional_effect",
                    "times_seen_singlemut": "times_seen"
                },
                inplace=True
            )
            df["selection"] = selection
            df["entry_condition"] = entry_condition
            df["label"] = label
            dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

# Load single-mutant data
func_effects_singlemut = load_func_effects_singlemut(entry_conditions)

# Convert times_seen to Int64
func_effects_singlemut["times_seen"] = func_effects_singlemut["times_seen"].astype("Int64")

print(f"Loaded {len(func_effects_singlemut):,} rows of single-mutant functional effects data")

Loaded 117,348 rows of single-mutant functional effects data


## Library comparison plots for single-mutant variants

Create scatter plots comparing LibA vs LibB for func-1 and func-2 separately, using only single-mutant variants.

In [6]:
print(f"\nLibrary comparison scatter plots (single-mutant only) for times_seen filter of {init_times_seen}\n")

# Create mutation identifier
func_effects_singlemut["mutation"] = (
    func_effects_singlemut["wildtype"] + 
    func_effects_singlemut["site"].astype(str) + 
    func_effects_singlemut["mutant"]
)

# Filter by times_seen
func_effects_singlemut_filtered = func_effects_singlemut.query("times_seen >= @init_times_seen")

# Create comparison plots for each entry condition and func type
plots_singlemut = []

for entry_condition in ["2-3 entry", "2-6 entry", "mix entry"]:
    entry_plots = []
    
    for func_type in ["func-1", "func-2"]:
        # Get data for LibA and LibB for this func type
        libA_label = f"LibA-{func_type}"
        libB_label = f"LibB-{func_type}"
        
        df = (
            func_effects_singlemut_filtered
            .query("entry_condition == @entry_condition")
            .query("label in [@libA_label, @libB_label]")
            .pivot_table(
                index="mutation",
                columns="label",
                values="functional_effect"
            )
            .reset_index()
        )
        
        # Check if we have both columns
        if libA_label not in df.columns or libB_label not in df.columns:
            print(f"Skipping {entry_condition} {func_type}: missing data")
            continue
        
        # Drop NaN values and calculate correlation
        df_clean = df[[libA_label, libB_label]].dropna()
        n = len(df_clean)
        r = df_clean.corr().values[1, 0] if n > 0 else 0
        
        print(f"{entry_condition} - {func_type}: n={n}, r={r:.3f}")
        
        if n == 0:
            continue
        
        # Create scatter plot
        chart = (
            alt.Chart(df)
            .add_params(mutation_selection)
            .encode(
                alt.X(
                    libA_label,
                    scale=alt.Scale(nice=False, padding=4),
                    title=f"LibA {func_type}"
                ),
                alt.Y(
                    libB_label,
                    scale=alt.Scale(nice=False, padding=4),
                    title=f"LibB {func_type}"
                ),
                size=alt.condition(mutation_selection, alt.value(80), alt.value(30)),
                color=alt.condition(
                    mutation_selection, alt.value("red"), alt.value("black")
                ),
                opacity=alt.condition(
                    mutation_selection, alt.value(1), alt.value(0.25)
                ),
                tooltip=[
                    "mutation",
                    alt.Tooltip(libA_label, format=".3f"),
                    alt.Tooltip(libB_label, format=".3f"),
                ],
            )
            .mark_circle()
            .properties(
                width=250,
                height=250,
                title=alt.TitleParams(
                    f"{entry_condition}\nR = {r:.2f}, N = {n}",
                    fontSize=12,
                    fontWeight="normal",
                ),
            )
        )
        
        entry_plots.append(chart)
    
    if entry_plots:
        plots_singlemut.append(alt.hconcat(*entry_plots))

# Display all plots
if plots_singlemut:
    final_chart_singlemut = (
        alt.vconcat(*plots_singlemut)
        .configure_axis(grid=False)
        .properties(title="Library Replicate Comparison: LibA vs LibB (Single-Mutant Variants Only)")
    )
    display(final_chart_singlemut)
else:
    print("No plots generated")


Library comparison scatter plots (single-mutant only) for times_seen filter of 2

2-3 entry - func-1: n=4277, r=0.873


2-3 entry - func-2: n=4222, r=0.868


2-6 entry - func-1: n=4277, r=0.924
2-6 entry - func-2: n=4222, r=0.924


mix entry - func-1: n=4277, r=0.919


mix entry - func-2: n=4222, r=0.916


alt.VConcatChart(...)